In [24]:
# menghubungkan google drive & menyiapkan struktur folder

from google.colab import drive
drive.mount("/content/drive")

import os
DIR_KERJA = "/content/data"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum2"

os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['transaksi_bersih_praktikum.csv', 'transaksi_mentah_praktikum.csv']


Menghubungkan DIR_KERJA dan DIR_SIMPAN dari google drive ke colab, supaya penyimpanan data dapat dilakukan dengan baik.

In [25]:
!pip install faker

Menginstall library faker

In [26]:
#k1
import numpy as np
import pandas as pd
from faker import Faker
import random

Melakukan import library dan inisialisasi untuk menyiapkan seluruh pustaka yang dipakai sepanjang praktikum ini.

In [27]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
  trx_id = f"TRX{i:05d}"
  nama_pelanggan = fake.name()
  produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", " "])
  kategori = random.choice(kategori_produk)
  harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
  qty = random.randint(1, 5)

  #variasi format harga : angka polos, ada "Rp", ada desimal ".0", ada spasi
  harga_variants = [
      str(harga_dasar),
      f"Rp{harga_dasar:,}".replace(",", "."),
      f"{harga_dasar}.0",
      f"{harga_dasar}",
  ]
  harga = random.choice(harga_variants)

  # variasi format tanggal : ISO, DD/MM/YYYY, DD-MM-YYYY
  tgl = fake.date_between(start_date="-90d", end_date="today")
  tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
  tanggal = random.choice(tgl_variants)

  metode = random.choice(metode_bayar)
  if random.random() < 0.3:
    metode = metode.lower()
  if random.random() < 0.2:
    kategori = kategori.upper() + " "

  kota = fake.city()
  rating = random.choice([1, 2, 3, 4, 5, None, None]) #rating opsional

  rows.append({
      "transaction_id": trx_id,
      "customer_name": nama_pelanggan,
      "product_name": produk.strip(),
      "category": kategori,
      "price": harga,
      "quatinty": qty,
      "payment_method": metode,
      "transaction_date": tanggal,
      "shipping_city": kota,
      "rating": rating,
  })

df = pd.DataFrame(rows)

# suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
  idx = df.sample(frac=frac, random_state=SEED).index
  df.loc[idx, col] = np.nan

# duplikasi 15 baris baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Save the CSV directly to the DIR_SIMPAN directory
output_file_path = os.path.join(DIR_SIMPAN, "transaksi_mentah_praktikum.csv")
df.to_csv(output_file_path, index=False)
print("Jumlah baris:", len(df))
print(f"File saved to: {output_file_path}")

Jumlah baris: 515
File saved to: /content/drive/MyDrive/BigData/Praktikum2/transaksi_mentah_praktikum.csv


Membuat dataset sintesis, mensimulasikan proses acquisition pada data transaksi. Terdapat beberapa bagian yang sengaja diberikan format harga yang beragam, tanggal, kapitalisasi, missing value, dan baris duplicate. Kondisi-kondisi tersebut meniru masalah yang marak muncul di data mentah dunia nyata. Juga melakukan penyimpanan data mentah csv pada penyimpanan google drive.

In [28]:
#k3
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quatinty              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


Deteksi dan penanganan missing value, pada output dapat terlihat bahwa pada customer_name terdapat 20 missing value, pada payment_method terdapat 16 missing value, pada shipping_city terdapat 30 missing value, dan pada rating terdapat 166 missing value.

In [29]:
# strategi penanganan

df = df.dropna(subset=["customer_name",  "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("tidak diketahui")
print("Baris:", len(df))

Baris: 495


Juga melakukan strategi penanganan untuk missing value tersebut, dengan menentukan per kolom dan jangan disamaratakan. Seperti untuk customer_name strateginya adalah dengan membuang baris (dropna), begitu juga dengan payment_method. Lalu untuk shipping_city strateginya direplace dengan “Tidak Diketahui”, dan kemudian untuk rating dibiarkan kosong.

Sehingga baris yang didapatkan adalah sebanyak 495 baris data setelah dilakukan pembersihan untuk missing value.

In [30]:
#k4
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


Deteksi dan penanganan duplicate. Pada bagian ini akan dilakukan pemantauan jumlah untuk data yang duplikat dan sekalian dengan penanganannya yaitu dengan melakukan drop pada baris duplikat, kemudian menghitung kembali jumlah baris setelah dilakukan pembersihan pada duplicate data.

In [31]:
#k5

#standarisasi teks kategorikal
for col in ["category", "payment_method", "shipping_city"]:
  df[col] = df[col].astype("string").str.strip().str.title()

# "cod" adalah singkatan, kembalikan ke huruf kapital penuh setelah title case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

# korreksi tipe data pada kolom price
def bersihkan_harga(x):
  if pd.isna(x):
    return np.nan
  x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
  try:
    return float(x)
  except ValueError:
    return np.nan

df["price"]  = df["price"].apply(bersihkan_harga)

# standarisasi format tanggal ke YYYY-MM-DD
def parse_tanggal(x):
  for fmt in ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"]:
    try:
      return pd.to_datetime(x, format=fmt)
    except ValueError:
      continue
  return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

#finalisasi tipe data
df["quatinty"] = df["quatinty"].astype(int)
df["price"] = df["price"].astype(float)

Melakukan pengoreksian tipe data dan standarisasi format.

Pertama, untuk standarisasi teks kategorikal (category, payment_method, shipping_city)

Kedua, melakukan koreksi tipe data pada kolom price (dari yang tadinya teks bercampur simbol, diubah sepenuhnya menjadi numerik, sehingga mudah dilakukan perhitungan)

Ketiga, lakukan standarisasi formata tanggal ke YYYY-MM-DD, cara terbaik dan paling aman adalah dengan mencoba format eksplisit satu per satu untuk setiap nilai.

Keempat, finalisasikan tipe data, pada kode di bawah ini, quantity menjadi integer dana price menjadi float.

In [32]:
#k6
path_to_clean = os.path.join(DIR_SIMPAN, "transaksi_bersih_praktikum.csv")
df.to_csv(path_to_clean, index=False)
print("Dataset bersih tersimpan:", len(df), "baris")
print(f"File saved to: {path_to_clean}")

Dataset bersih tersimpan: 490 baris
File saved to: /content/drive/MyDrive/BigData/Praktikum2/transaksi_bersih_praktikum.csv


Lakukan ekspor dataset yang bersih ke folder penyimpanan dan mendeteksi jumlah baris bersih setelah dilakukan pra processing data, yaitu sebanyak 490 baris.
